In [ ]:
import pickle, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

# ===== Load WESAD Subject (S16.pkl) =====
with open("/content/drive/MyDrive/PD DATASETS/Stress PD/WESAD/S16/S16.pkl", "rb") as file:
    data = pickle.load(file, encoding="latin1")

chest_signals = data['signal']['chest']
signals = np.column_stack([np.array(chest_signals[k]) for k in chest_signals.keys()])
labels = np.array(data['label'])  # 0=baseline, 1=stress, 2=amusement, 3=meditation

scaler = StandardScaler()
X_stress = scaler.fit_transform(signals)
y_stress = (labels == 1).astype(int)  # 1=stress, 0=others

# ===== Downsample to save RAM =====
X_stress = X_stress[::10]
y_stress = y_stress[::10]
print("Stress dataset after downsampling →", X_stress.shape, y_stress.shape)

# ===== Make Sequences =====
SEQ_LEN = 30
X_seq, y_seq = [], []
for i in range(len(X_stress) - SEQ_LEN):
    X_seq.append(X_stress[i:i+SEQ_LEN])
    y_seq.append(y_stress[i+SEQ_LEN])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# ===== LSTM =====
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, X_train_s.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_s, y_train_s, validation_data=(X_test_s, y_test_s), epochs=3, batch_size=32, verbose=1)
_, acc_stress_lstm = model_lstm.evaluate(X_test_s, y_test_s, verbose=0)

# ===== GRU =====
model_gru = Sequential([
    GRU(64, input_shape=(SEQ_LEN, X_train_s.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.fit(X_train_s, y_train_s, validation_data=(X_test_s, y_test_s), epochs=3, batch_size=32, verbose=1)
_, acc_stress_gru = model_gru.evaluate(X_test_s, y_test_s, verbose=0)

print("📊 WESAD Results → LSTM:", acc_stress_lstm, " GRU:", acc_stress_gru)


Stress dataset after downsampling → (394170, 8) (394170,)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
9854/9854 ━━━━━━━━━━━━━━━━━━━━ 199s 20ms/step - accuracy: 0.9795 - loss: 0.0526 - val_accuracy: 0.9973 - val_loss: 0.0061
Epoch 2/3
9854/9854 ━━━━━━━━━━━━━━━━━━━━ 222s 22ms/step - accuracy: 0.9963 - loss: 0.0098 - val_accuracy: 0.9968 - val_loss: 0.0065
Epoch 3/3
9854/9854 ━━━━━━━━━━━━━━━━━━━━ 195s 20ms/step - accuracy: 0.9971 - loss: 0.0075 - val_accuracy: 0.9982 - val_loss: 0.0037
Epoch 1/3
9854/9854 ━━━━━━━━━━━━━━━━━━━━ 241s 24ms/step - accuracy: 0.9813 - loss: 0.0510 - val_accuracy: 0.9967 - val_loss: 0.0070
Epoch 2/3
9236/9854 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9959 - loss: 0.0107

In [ ]:
# ===== Load Gait Excel =====
import pandas as pd
import numpy as np # Added numpy import
from sklearn.preprocessing import MinMaxScaler # Added MinMaxScaler import
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

df_gait = pd.read_excel("/content/demographics (1).xls")

y_gait = (df_gait['Group'] == "PD").astype(int)
X_gait = df_gait.drop(['ID','Study','Group','Gender'], axis=1)
X_gait = X_gait.fillna(X_gait.mean())

scaler = MinMaxScaler()
X_gait = scaler.fit_transform(X_gait)

SEQ_LEN = 10
X_seq, y_seq = [], []
for i in range(len(X_gait) - SEQ_LEN):
    X_seq.append(X_gait[i:i+SEQ_LEN])
    y_seq.append(y_gait.iloc[i+SEQ_LEN])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# ===== LSTM =====
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, X_train_g.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_g, y_train_g, validation_data=(X_test_g, y_test_g), epochs=3, batch_size=16, verbose=1)
_, acc_gait_lstm = model_lstm.evaluate(X_test_g, y_test_g, verbose=0)

# ===== GRU =====
model_gru = Sequential([
    GRU(64, input_shape=(SEQ_LEN, X_train_g.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.fit(X_train_g, y_train_g, validation_data=(X_test_g, y_test_g), epochs=3, batch_size=16, verbose=1)
_,_

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.5854 - loss: 0.6833 - val_accuracy: 0.6875 - val_loss: 0.6431
Epoch 2/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6759 - loss: 0.6504 - val_accuracy: 0.8750 - val_loss: 0.5948
Epoch 3/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8016 - loss: 0.5912 - val_accuracy: 0.9375 - val_loss: 0.5261
Epoch 1/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.6182 - loss: 0.6745 - val_accuracy: 0.7500 - val_loss: 0.6280
Epoch 2/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6632 - loss: 0.6285 - val_accuracy: 0.8750 - val_loss: 0.5921
Epoch 3/3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7391 - loss: 0.6038 - val_accuracy: 0.8750 - val_loss: 0.5499


(0.526059091091156, 0.526059091091156)

In [ ]:
# ===== Load Voice Excel =====
import pandas as pd
import numpy as np # Added numpy import
from sklearn.preprocessing import StandardScaler # Added StandardScaler import
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

df_voice = pd.read_excel("/content/drive/MyDrive/PD DATASETS/VOICE PD/pd 234 voice.xlsx")

y_voice = (df_voice['total_UPDRS'] > df_voice['total_UPDRS'].median()).astype(int)
X_voice = df_voice.drop(['subject#','test_time','motor_UPDRS','total_UPDRS'], axis=1)

scaler = StandardScaler()
X_voice = scaler.fit_transform(X_voice)

# ===== Downsample =====
X_voice = X_voice[::2]
y_voice = y_voice[::2]

SEQ_LEN = 20
X_seq, y_seq = [], []
for i in range(len(X_voice) - SEQ_LEN):
    X_seq.append(X_voice[i:i+SEQ_LEN])
    y_seq.append(y_voice.iloc[i+SEQ_LEN])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

# ===== LSTM =====
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, X_train_v.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.fit(X_train_v, y_train_v, validation_data=(X_test_v, y_test_v), epochs=3, batch_size=16, verbose=1)
_, acc_voice_lstm = model_lstm.evaluate(X_test_v, y_test_v, verbose=0)

# ===== GRU =====
model_gru = Sequential([
    GRU(64, input_shape=(SEQ_LEN, X_train_v.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gru.fit(X_train_v, y_train_v, validation_data=(X_test_v, y_test_v), epochs=3, batch_size=16, verbose=1)
_, acc_voice_gru = model_gru.evaluate(X_test_v, y_test_v, verbose=0)

print("📊 Voice Results → LSTM:", acc_voice_lstm, " GRU:", acc_voice_gru)

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


146/146 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.6351 - loss: 0.6152 - val_accuracy: 0.7928 - val_loss: 0.4297
Epoch 2/3
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8411 - loss: 0.3900 - val_accuracy: 0.8408 - val_loss: 0.3714
Epoch 3/3
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8706 - loss: 0.3132 - val_accuracy: 0.9041 - val_loss: 0.2530
Epoch 1/3
146/146 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.6367 - loss: 0.6369 - val_accuracy: 0.7551 - val_loss: 0.4866
Epoch 2/3
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7778 - loss: 0.4691 - val_accuracy: 0.8134 - val_loss: 0.3772
Epoch 3/3
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8387 - loss: 0.3507 - val_accuracy: 0.8408 - val_loss: 0.3352
📊 Voice Results → LSTM: 0.9041095972061157  GRU: 0.840753436088562


In [ ]:
# ===== WESAD (Stress Dataset) =====
import pickle
with open("/content/drive/MyDrive/PD DATASETS/Stress PD/WESAD/S16/S16.pkl", "rb") as file:
    data = pickle.load(file, encoding="latin1")

chest_signals = data['signal']['chest']
signals = np.column_stack([np.array(chest_signals[k]) for k in chest_signals.keys()])
labels = np.array(data['label'])

scaler = StandardScaler()
X_stress = scaler.fit_transform(signals)
y_stress = (labels == 1).astype(int)

# Downsample for memory
X_stress_ds = X_stress[::10]
y_stress_ds = y_stress[::10]

print("Stress downsampled →", X_stress_ds.shape, y_stress_ds.shape)

Stress downsampled → (394170, 8) (394170,)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

# ==============================================
# Step 1: Downsample (memory safe)
# ==============================================
X_stress_ds = X_stress[::10]   # Stress already heavy
y_stress_ds = y_stress[::10]

X_voice_ds = X_voice[::2]      # Reduce by half
y_voice_ds = y_voice[::2]

X_gait_ds = X_gait[::2]        # Light downsample
y_gait_ds = y_gait.iloc[::2].reset_index(drop=True)

print("Downsampled Shapes:")
print("Stress →", X_stress_ds.shape, y_stress_ds.shape)
print("Gait   →", X_gait_ds.shape, y_gait_ds.shape)
print("Voice  →", X_voice_ds.shape, y_voice_ds.shape)

# ==============================================
# Step 2: Align dataset sizes
# ==============================================
min_len = min(len(X_stress_ds), len(X_gait_ds), len(X_voice_ds))

X_fusion = np.hstack([
    X_stress_ds[:min_len],
    X_gait_ds[:min_len],
    X_voice_ds[:min_len]
])
y_fusion = y_gait_ds[:min_len].values   # PD vs Control labels

print("Fusion dataset →", X_fusion.shape, y_fusion.shape)

# ==============================================
# Step 3: Make Sequences
# ==============================================
SEQ_LEN = 30
X_seq, y_seq = [], []
for i in range(len(X_fusion) - SEQ_LEN):
    X_seq.append(X_fusion[i:i+SEQ_LEN])
    y_seq.append(y_fusion[i+SEQ_LEN])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

print("Fusion sequences →", X_train_f.shape, X_test_f.shape)

# ==============================================
# Step 4: LSTM Model
# ==============================================
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, X_train_f.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n🚀 Training LSTM on Fusion dataset...")
model_lstm.fit(X_train_f, y_train_f, validation_data=(X_test_f, y_test_f),
               epochs=3, batch_size=32, verbose=1)
_, acc_fusion_lstm = model_lstm.evaluate(X_test_f, y_test_f, verbose=0)

# ==============================================
# Step 5: GRU Model
# ==============================================
model_gru = Sequential([
    GRU(64, input_shape=(SEQ_LEN, X_train_f.shape[2])),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n🚀 Training GRU on Fusion dataset...")
model_gru.fit(X_train_f, y_train_f, validation_data=(X_test_f, y_test_f),
              epochs=3, batch_size=32, verbose=1)
_, acc_fusion_gru = model_gru.evaluate(X_test_f, y_test_f, verbose=0)

# ==============================================
# Step 6: Final Results
# ==============================================
print("=======================================")
print("📊 Fusion Dataset Results (Downsampled)")
print("LSTM Accuracy:", acc_fusion_lstm)
print("GRU  Accuracy:", acc_fusion_gru)
print("=======================================")


Downsampled Shapes:
Stress → (394170, 8) (394170,)
Gait   → (83, 10) (83,)
Voice  → (1469, 18) (1469,)
Fusion dataset → (83, 36) (83,)
Fusion sequences → (42, 30, 36) (11, 30, 36)

🚀 Training LSTM on Fusion dataset...
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 599ms/step - accuracy: 0.4578 - loss: 0.7117 - val_accuracy: 0.8182 - val_loss: 0.6504
Epoch 2/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.7158 - loss: 0.6220 - val_accuracy: 0.8182 - val_loss: 0.5785
Epoch 3/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.8790 - loss: 0.5428 - val_accuracy: 0.8182 - val_loss: 0.5202

🚀 Training GRU on Fusion dataset...
Epoch 1/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 428ms/step - accuracy: 0.4315 - loss: 0.7433 - val_accuracy: 0.8182 - val_loss: 0.6309
Epoch 2/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.7842 - loss: 0.5692 - val_accuracy: 0.8182 - val_loss: 0.5792
Epoch 3/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.7054 - loss: 0.5601 - val_accuracy: 0.8182 - val_loss: 0.5501
📊 Fusion Dataset Results (Downsampled)
LSTM Accuracy: 0.8181818127632141
GRU  Accuracy: 0.8181818127632141


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

# ==============================================
# Hybrid Model (LSTM + GRU)
# ==============================================
model_hybrid = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, X_train_f.shape[2])),
    GRU(64),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_hybrid.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n🚀 Training Hybrid (LSTM+GRU) on Fusion dataset...")
history_hybrid = model_hybrid.fit(
    X_train_f, y_train_f,
    validation_data=(X_test_f, y_test_f),
    epochs=3, batch_size=32, verbose=1
)

_, acc_fusion_hybrid = model_hybrid.evaluate(X_test_f, y_test_f, verbose=0)

print("✅ Hybrid Fusion Accuracy:", acc_fusion_hybrid)



🚀 Training Hybrid (LSTM+GRU) on Fusion dataset...
Epoch 1/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 584ms/step - accuracy: 0.3690 - loss: 0.7083 - val_accuracy: 0.8182 - val_loss: 0.6165
Epoch 2/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.8472 - loss: 0.6019 - val_accuracy: 0.8182 - val_loss: 0.5489
Epoch 3/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.8631 - loss: 0.5234 - val_accuracy: 0.8182 - val_loss: 0.4819
✅ Hybrid Fusion Accuracy: 0.8181818127632141


In [ ]:
# Save Hybrid Model
model_hybrid.save("/content/fusion_hybrid_model.h5")
model_hybrid.save("/content/fusion_hybrid_model.keras")

print("✅ Hybrid model saved successfully!")


✅ Hybrid model saved successfully!
